In [26]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [27]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [28]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [29]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from [https://ollama.com/download](https://ollama.com/download) for your operating system:
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test the local server, run:
   ```bash
   curl http://localhost:11434
   ```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```

and then call:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [30]:
print(assistant.rag("How do I run Olama locally?"))

The FAQ doesn’t mention **Olama** specifically. It does say you can run the course locally if you’re comfortable setting up the needed tools, including **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**.

If you meant a local setup for one of the course modules, you’ll need to document your setup and keep the environment reproducible.


In [31]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes — in most cases, you can join a course after it has started, as long as:\n\n- there are still spots available\n- the course is still accepting new students\n- you can catch up on any missed material\n\nIf you want, I can help you figure out the best next step. Just send me:\n- the course name\n- where you found it\n- whether you mean an online or in-person course\n\nIf you’re asking about a specific course, I can help you draft a message to the instructor or organizer.'

In [32]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [33]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [34]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join"}', call_id='call_P7EG23FMehL4HynyzDPz5b4h', name='search', type='function_call', id='fc_0d70bf358e6b79a9006a7cc43ea50c819b99fba016b068f009', caller=None, namespace=None, status='completed')]

In [35]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [36]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "5cc511f85b",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Does the course certificate show the number of course hours?",
    "answer": "No. The certificate does not state a total number of hours."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer r

In [37]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [38]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can join the course even if you just discovered it.

If you want a certificate, the key thing is to submit your project while submissions are still open.


In [39]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(781, 38)

In [40]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [41]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.First perform search, analyze the results 
and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [42]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [43]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)
print(response.output)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

[ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered the course can I join"}', call_id='call_c4EtZnAvwno5XMmcpDW6bAZh', name='search', type='function_call', id='fc_0f5feb9d545954ff006a7cc440a81c81988b65ff1f788365a9', caller=None, namespace=None, status='completed')]
function_call: search {"query":"join course enrollment discovered the course can I join"}


In [44]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [45]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local model Ollama"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 chat llama3 local server restart connection refused"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model and starts a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response indicating the server is up.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(re

'To run Ollama locally:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and starts a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response indicating the server is up.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show you how to use Ollama

In [46]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit definition chess opening"}
function_call: search {"query":"queen gambit chess move d4 d5 c4"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to gain control of the center and pull Black’s **d-pawn** away from the center.

A few key points:
- It’s called a **gambit** because White is temporarily “offering” a pawn.
- It’s called **Queen’s** because it begins with the **queen’s pawn**: the pawn in front of the queen.
- It’s one of the oldest and most respected openings in chess.

There are two main types:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain:
- the **idea behind the Queen’s Gambit**
- the difference between **Accepted** and 

'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to gain control of the center and pull Black’s **d-pawn** away from the center.\n\nA few key points:\n- It’s called a **gambit** because White is temporarily “offering” a pawn.\n- It’s called **Queen’s** because it begins with the **queen’s pawn**: the pawn in front of the queen.\n- It’s one of the oldest and most respected openings in chess.\n\nThere are two main types:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain:\n- the **idea behind the Queen’s Gambit**\n- the difference between **Accepted** and **Declined**\n- or give you a **simple move-by-move example**'

In [47]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening"}
iteration #3...
function_call: search {"query":"gambit chess opening queen's gambit FAQ"}
iteration #4...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen’s gambit,” so I can’t answer that from the course materials.

If you meant a course-related term, feel free to rephrase it with more context. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen’s gambit,” so I can’t answer that from the course materials.\n\nIf you meant a course-related term, feel free to rephrase it with more context. Are there other areas you want to explore?'

In [48]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [51]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [52]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [53]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [54]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [55]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [56]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received
